# Tools

A model's capabilities is extended with tools. Tools can be built into the API or accessed through functions and Model Context Protocol (MCP) services.

Tools include:

- Image generation.
- Web search.
- Function calling.
- Remote MCP Servers.
- File search.
- Code interpreter.
- Computer use.

# Function Calling

In this notebook we use function calling to illustrate tools and their usage. 

A useful resource is OpenAI's [Function Calling documentation for the responses API](https://platform.openai.com/docs/guides/function-calling?api-mode=responses).

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
import json

client = get_client()

# Tool Calling Flow

The idea behind using tools is that a model will request to use a tool. When a model examines a prompt, it may determine that to follow the instruction it should use a tool. It then generates a special type of output, which we will catch and call a specific tool. 

The logic flows as follows:


1. Make a request to the model with tools it could call: the model is given a prompt that can be processed with a tool at the model's disposal. e.g. here the model can access the tool get_weather as long as it is provided with the parameter 'location'
2. Receive a tool call from the model: the model identifies the location from the prompt as Paris. It calls the tool and supplies location as the parameter. 
3. Execute code on the application side with input from the tool call: the developer (user's client) intercepts the tool call and executes it (the command get_weather("paris")), and receives the answer from the tool.
the client side intercepts the runs the tool call and receives the output temperature = 14.
4. Make a second request to the model with the tool output: all prior messages along with the tool output is supplied to the model.
5. Receive a final response from the model (or more tool calls): the model compiles this to return the answer in a conversational format. 


<img src="./img/05_function-calling-diagram-steps.png" height=700>

# Make a request to the model with tools it could call

The tool definition includes the function name, its parameters, and additional metadata. 

In [3]:
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get horoscope for a given zodiac sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "zodiac_sign": {
                    "type": "string",
                    "description": "Zodiac sign e.g. Aries, Taurus",
                }
            },
            "required": ["zodiac_sign"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

def get_horoscope(zodiac_sign: str) -> str:
    # Dummy implementation for illustration
    horoscope = f"{zodiac_sign}: Today is a great day for new beginnings."
    return horoscope



We will need to retain some "memory" of the conversation. In this case, we will retain the message history and send it as context in our interaction with the model

In [4]:
input_list = [
    {"role": "user", "content": "What is my horoscope? I am an Aquarius."}
]

We prompt the model with the initial input.

In [5]:
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)

# Execute code on the application side with input from the tool call


Examine the response output. Notice that we find a "reasoning item" and a "function tool call".

In [6]:
[o.model_dump() for o in response.output]

[{'id': 'rs_01ffb3d599c220c7006a4c09ae5c40819d976b4a6df7f8c3e0',
  'summary': [],
  'type': 'reasoning',
  'content': [],
  'encrypted_content': None,
  'status': None},
 {'arguments': '{"zodiac_sign":"Aquarius"}',
  'call_id': 'call_ZqzaGVADnYrqRUjBFBaceAG5',
  'name': 'get_horoscope',
  'type': 'function_call',
  'id': 'fc_01ffb3d599c220c7006a4c09afc7ac819d88ad98591d580094',
  'namespace': None,
  'status': 'completed'}]

The function call item indicates that the model is requesting to run the function call: `get_horoscope(zodiac_sign='Aquarius')`. That's because the model extracted the zodiac from the user prompt based on the context, and is now supplying it as the parameter for the function (tool) `get_horoscope`.

In [7]:
response.output[1].model_dump()

{'arguments': '{"zodiac_sign":"Aquarius"}',
 'call_id': 'call_ZqzaGVADnYrqRUjBFBaceAG5',
 'name': 'get_horoscope',
 'type': 'function_call',
 'id': 'fc_01ffb3d599c220c7006a4c09afc7ac819d88ad98591d580094',
 'namespace': None,
 'status': 'completed'}

In the code below:

+ Execute the function logic for get_horoscope.
+ Add the function output to the input_list.

In [8]:
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_horoscope":
            # Execute the function logic for get_horoscope
            horoscope = get_horoscope(**json.loads(item.arguments))
            
            # Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "horoscope": horoscope
                })
            })

# Make a second request to the model with the tool output


Examine the final input_list which is the context of model call.

In [9]:
[e for e in input_list]

[{'role': 'user', 'content': 'What is my horoscope? I am an Aquarius.'},
 ResponseReasoningItem(id='rs_01ffb3d599c220c7006a4c09ae5c40819d976b4a6df7f8c3e0', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"zodiac_sign":"Aquarius"}', call_id='call_ZqzaGVADnYrqRUjBFBaceAG5', name='get_horoscope', type='function_call', id='fc_01ffb3d599c220c7006a4c09afc7ac819d88ad98591d580094', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_ZqzaGVADnYrqRUjBFBaceAG5',
  'output': '{"horoscope": "Aquarius: Today is a great day for new beginnings."}'}]

note that call_id appears twice in the output above. that's because the same call_id is attached to both the model call and the response from the model. 

In [10]:
response = client.responses.create(
    model="gpt-5",
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)


# Receive a final response from the model (or more tool calls)

In [11]:
response.output_text

'Aquarius: Today is a great day for new beginnings.'

In [12]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01ffb3d599c220c7006a4c09b07530819d84294d60e2ae29e5",
  "created_at": 1783368112.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a horoscope generated by a tool.",
  "metadata": {},
  "model": "gpt-5-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "msg_01ffb3d599c220c7006a4c09b0e4ac819d8c6f171f6354ba7e",
      "content": [
        {
          "annotations": [],
          "text": "Aquarius: Today is a great day for new beginnings.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_horoscope",
      "parameters": {
        "type": "object",
        "properties": {
          "zodiac_sign": {
            "type": "string",
            "description": "Zodiac sign e.g